# Debug 03: same function, two run names

This notebook compares two SMAC runs created by the exact same Python function. The only intended difference is the `run_label`, which changes the scenario name/output directory.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

In [ ]:
ROOT = Path("/home/io632776/experiments/adaptive-smac/experiments/hartmann/Debug_03")
RESULT_DIRECTORY = ROOT / "same_function_two_names"
RUN_LABELS = ["same_function_a", "same_function_b"]
SEED = 0
RUN_DIRS = {label: RESULT_DIRECTORY / label / str(SEED) for label in RUN_LABELS}
FIGURE_DIRECTORY = ROOT / "figures"
FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)

for label, run_dir in RUN_DIRS.items():
    print(label, run_dir, run_dir.exists())

## Helpers

In [ ]:
def load_json(path: Path) -> dict:
    with open(path) as fh:
        return json.load(fh)


def ordered_trials(run_dir: Path) -> list[dict]:
    runhistory = load_json(run_dir / "runhistory.json")
    configs = {str(config_id): config for config_id, config in runhistory["configs"].items()}
    origins = {str(config_id): origin for config_id, origin in runhistory.get("config_origins", {}).items()}
    rows = sorted(runhistory["data"], key=lambda row: (row["starttime"], row["endtime"]))

    ordered = []
    for trial_number, row in enumerate(rows, start=1):
        config_id = str(row["config_id"])
        ordered.append(
            {
                "trial": trial_number,
                "config_id": config_id,
                "config": configs[config_id],
                "cost": float(row["cost"]),
                "origin": origins.get(config_id),
            }
        )
    return ordered


def best_so_far(costs):
    return np.minimum.accumulate(np.asarray(costs, dtype=float))

## Scenario metadata

In [ ]:
metadata_rows = []
for label, run_dir in RUN_DIRS.items():
    scenario = load_json(run_dir / "scenario.json")
    meta = scenario.get("_meta", {})
    model = meta.get("model", {})
    initial_design = meta.get("initial_design", {})
    metadata_rows.append(
        {
            "run": label,
            "name": scenario.get("name"),
            "output_directory": scenario.get("output_directory"),
            "n_trials": scenario.get("n_trials"),
            "seed": scenario.get("seed"),
            "initial_design_n_configs": initial_design.get("n_configs"),
            "model_min_samples_leaf": model.get("min_samples_leaf"),
            "model_max_depth": model.get("max_depth"),
            "model_random_state": model.get("random_state"),
        }
    )

pd.DataFrame(metadata_rows)

## Trial-by-trial comparison

In [ ]:
trials = {label: ordered_trials(run_dir) for label, run_dir in RUN_DIRS.items()}
a = trials[RUN_LABELS[0]]
b = trials[RUN_LABELS[1]]

comparison_rows = []
first_difference = None
for trial_a, trial_b in zip(a, b):
    same_config = trial_a["config"] == trial_b["config"]
    same_cost = np.isclose(trial_a["cost"], trial_b["cost"], rtol=0, atol=1e-15)
    if first_difference is None and (not same_config or not same_cost):
        first_difference = trial_a["trial"]

    comparison_rows.append(
        {
            "trial": trial_a["trial"],
            "same_config": same_config,
            "same_cost": bool(same_cost),
            f"{RUN_LABELS[0]}_cost": trial_a["cost"],
            f"{RUN_LABELS[1]}_cost": trial_b["cost"],
            f"{RUN_LABELS[0]}_origin": trial_a["origin"],
            f"{RUN_LABELS[1]}_origin": trial_b["origin"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
print("first_difference:", first_difference)
comparison_df.head(35)

In [ ]:
if first_difference is not None:
    display(comparison_df[comparison_df["trial"].between(max(1, first_difference - 5), first_difference + 5)])
    config_a = a[first_difference - 1]["config"]
    config_b = b[first_difference - 1]["config"]
    diff_df = pd.DataFrame({RUN_LABELS[0]: config_a, RUN_LABELS[1]: config_b})
    diff_df["abs_diff"] = (diff_df[RUN_LABELS[0]] - diff_df[RUN_LABELS[1]]).abs()
    display(diff_df)
else:
    print("Runs are identical trial-by-trial.")

## Best-so-far trajectory plot

In [ ]:
plt.figure(figsize=(12, 6))
for label in RUN_LABELS:
    costs = np.asarray([trial["cost"] for trial in trials[label]])
    trajectory = best_so_far(costs)
    plt.plot(np.arange(1, len(trajectory) + 1), trajectory, label=label, linewidth=2.4)

if first_difference is not None:
    plt.axvline(first_difference, color="red", linestyle="--", label=f"first difference: {first_difference}")
plt.title("Same SMAC function, different scenario names")
plt.xlabel("Trial number")
plt.ylabel("Best incumbent cost so far")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIRECTORY / "same_function_two_names_best_so_far.png", dpi=200)
plt.show()

## Raw cost plot

In [ ]:
plt.figure(figsize=(12, 5))
for label in RUN_LABELS:
    costs = np.asarray([trial["cost"] for trial in trials[label]])
    plt.plot(np.arange(1, len(costs) + 1), costs, label=label, alpha=0.8)

if first_difference is not None:
    plt.axvline(first_difference, color="red", linestyle="--", label=f"first difference: {first_difference}")
plt.title("Raw trial costs: same function, two names")
plt.xlabel("Trial number")
plt.ylabel("Cost")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIRECTORY / "same_function_two_names_raw_costs.png", dpi=200)
plt.show()